# Price forecast: spot + total bill (next day / week / month)

## 1 · Config + helpers
Sets up the Postgres connection (`ENGINE`) and the model hyperparameters.
`sql()` reads a query into a DataFrame; `lit()` formats a Python value as a SQL
literal for the insert batches; `fit()` builds the gradient-boosting model.
Everything below depends on this cell.

In [ ]:
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sqlalchemy import create_engine, text

ROOT = Path.cwd()
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)

ENGINE = create_engine("postgresql+psycopg2://mlinggar:@localhost:5432/swedish_el")

VAT_RATE = 0.25
TAX_CATEGORY, FEE_CUSTOMER, FEE_WEIGHTING = "DA", "1", "viktat"
HOLDOUT_DAYS, FOLDS, MAX_LEAD, SEED = 60, 6, 62, 42

FEATS = ["lead_days", "dow_of_target", "month_of_target", "price_prevday",
         "price_mean_prev7", "temp_prevday", "temp_max_prevday", "wind_prevday",
         "precip_prevday", "discharge_prevday", "gas_price_sek_kwh", "zone_idx"]
FEATS_AGG = ["dow_of_cutoff", "month_of_period", "price_prevday", "price_mean_prev7",
             "temp_prevday", "temp_max_prevday", "wind_prevday", "precip_prevday",
             "discharge_prevday", "gas_price_sek_kwh", "zone_idx"]
WEATHER_COLS = ["temp_prevday", "temp_max_prevday", "wind_prevday", "precip_prevday"]


def sql(statement):
    return pd.read_sql(text(statement), ENGINE)


def lit(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NULL"
    if isinstance(v, (bool, np.bool_)):
        return "true" if v else "false"
    if isinstance(v, (int, float, np.integer, np.floating)):
        return str(v)
    return "'" + str(v).replace("'", "''") + "'"


def fit(df, feats, label):
    m = HistGradientBoostingRegressor(max_iter=100, learning_rate=0.1,
                                      max_depth=6, random_state=SEED)
    m.fit(df[feats], df[label])
    return m


print("[config] ready")

## 2 · Fetch
Pulls the three inputs from Postgres: daily facts (`gold.fact_daily`), gas prices
(`gold.gas_price_daily_sek`), and consumer tax/fee (`silver.consumer_prices`).
Also caches each to `data/*.csv` so you can inspect or re-run without re-querying.

In [ ]:
def fetch():
    fact = sql("SELECT zone_id AS zone, date_local::text AS d, price_sek_kwh_avg,"
               " temp_c_avg, temp_c_max, wind_speed_ms_avg, precipitation_mm_sum, discharge_m3s"
               " FROM gold.fact_daily")
    gas = sql("SELECT gas_date::text AS gd, gas_price_sek_kwh AS gp"
              " FROM gold.gas_price_daily_sek")
    cp = sql("SELECT table_name, customer_category, weighting, period_start::text AS period_start,"
             " elskatt_sek_kwh, elnatspris_sek_kwh"
             " FROM silver.consumer_prices")
    fact.to_csv(DATA / "fact.csv", index=False)
    gas.to_csv(DATA / "gas.csv", index=False)
    cp.to_csv(DATA / "cp.csv", index=False)
    print(f"[fetch] fact={len(fact)} gas={len(gas)} cp={len(cp)}")
    return fact, gas, cp

fact, gas, cp = fetch()

## 3 · Build dataset
Turns the daily facts into a supervised learning table: one row per (zone,
cutoff-date, lead-day). Features are lagged (prev-day / prev-7-day) so nothing
leaks the future; gas is joined as-of the cutoff. Produces `sk` (all rows) and
`labeled` (rows with a known outcome) plus a leakage assertion.

In [ ]:
def build(fact, gas):
    fact["d"] = pd.to_datetime(fact["d"])
    day = fact.sort_values(["zone", "d"]).reset_index(drop=True)
    g = day.groupby("zone")
    for src, dst in [("price_sek_kwh_avg", "price_prevday"), ("temp_c_avg", "temp_prevday"),
                     ("temp_c_max", "temp_max_prevday"), ("wind_speed_ms_avg", "wind_prevday"),
                     ("precipitation_mm_sum", "precip_prevday"), ("discharge_m3s", "discharge_prevday")]:
        day[dst] = g[src].shift(1)
    day["price_mean_prev7"] = g["price_sek_kwh_avg"].transform(lambda x: x.rolling(7).mean()).shift(1)
    day["cutoff_date"] = day["d"]
    day["zone_idx"] = day["zone"].map({"SE1": 0.0, "SE2": 1.0, "SE3": 2.0}).fillna(3.0)

    cutoffs = day[["zone", "cutoff_date"]]
    leads = pd.DataFrame({"lead_days": range(1, MAX_LEAD + 1)})
    sk = cutoffs.merge(leads, how="cross")
    sk["target_date"] = sk["cutoff_date"] + pd.to_timedelta(sk["lead_days"], unit="D")

    lab = day[["zone", "d", "price_sek_kwh_avg"]].rename(
        columns={"d": "target_date", "price_sek_kwh_avg": "target_price_sek_kwh"})
    lab7 = day[["zone", "d", "price_sek_kwh_avg"]].copy()
    lab7["target_date"] = lab7["d"] + pd.Timedelta(days=7)
    lab7 = lab7.rename(columns={"price_sek_kwh_avg": "naive_week"})[["zone", "target_date", "naive_week"]]

    feat_cols = ["zone", "cutoff_date", "price_prevday", "price_mean_prev7", "temp_prevday",
                 "temp_max_prevday", "wind_prevday", "precip_prevday", "discharge_prevday", "zone_idx"]
    sk = sk.merge(lab, on=["zone", "target_date"], how="left")
    sk = sk.merge(lab7, on=["zone", "target_date"], how="left")
    sk = sk.merge(day[feat_cols], on=["zone", "cutoff_date"], how="left")

    gas["gd"] = pd.to_datetime(gas["gd"])
    sk["_gdt"] = sk["cutoff_date"] - pd.Timedelta(days=1)
    sk = pd.merge_asof(sk.sort_values("_gdt"), gas.sort_values("gd"),
                       left_on="_gdt", right_on="gd", allow_exact_matches=True)
    sk = sk.rename(columns={"gp": "gas_price_sek_kwh"}).drop(columns=["_gdt", "gd"])

    sk["dow_of_target"] = sk["target_date"].dt.weekday
    sk["month_of_target"] = sk["target_date"].dt.month
    sk["lead_bucket"] = np.where(sk["lead_days"] == 1, "1",
                                 np.where(sk["lead_days"] <= 8, "2-8", "9-62"))
    sk["week_start"] = sk["cutoff_date"] - pd.to_timedelta(sk["cutoff_date"].dt.weekday, "D") + pd.Timedelta(days=7)
    sk["week_end"] = sk["week_start"] + pd.Timedelta(days=6)
    sk["month_start"] = sk["cutoff_date"] + pd.offsets.MonthBegin(1)
    sk["month_end"] = sk["month_start"] + pd.offsets.MonthEnd(1)

    labeled = sk[sk["target_price_sek_kwh"].notna()].copy()
    assert (sk["cutoff_date"] < sk["target_date"]).all(), "FAIL: leakage"
    assert len(labeled) > 0, "FAIL: empty dataset"
    print(f"[data] rows={len(sk)} labeled={len(labeled)}")
    return day, sk, labeled

day, sk, labeled = build(fact, gas)

## 4 · Backtest
Walk-forward validation across `FOLDS` 30-day windows ending in a 60-day holdout.
Compares the GBT model against 3 naive baselines (previous-day, same-weekday,
seasonal mean) using MAE and hit-rate, per lead-bucket and per zone. Populates
`ev_rows` (the evidence table) and prints the MAE comparison.

In [ ]:
mx = labeled["target_date"].max()
end0 = mx + pd.Timedelta(days=1)
hold_start = mx - pd.Timedelta(days=HOLDOUT_DAYS)
bounds = [end0, hold_start]
cursor = hold_start
for _ in range(FOLDS):
    cursor -= pd.Timedelta(days=30)
    bounds.append(cursor)

ev_rows = []
now = dt.datetime.now().isoformat(sep=" ", timespec="seconds")

for i in range(FOLDS):
    test_start, test_end = bounds[i + 1], bounds[i]
    is_hold = (i == 0)
    train = labeled[(labeled["target_date"] < test_start) & (labeled["cutoff_date"] < test_start)]
    test = labeled[(labeled["target_date"] >= test_start) & (labeled["target_date"] < test_end)
                   & (labeled["cutoff_date"] < test_start)].copy()
    if len(test) == 0 or len(train) == 0:
        continue
    season = (train.groupby(["month_of_target", "dow_of_target"])["target_price_sek_kwh"]
              .mean().rename("seasonal_mean").reset_index())
    model = fit(train, FEATS, "target_price_sek_kwh")
    test["gbt"] = model.predict(test[FEATS])
    test = test.merge(season, on=["month_of_target", "dow_of_target"], how="left")
    for m in ["gbt", "price_prevday", "naive_week", "seasonal_mean"]:
        sub = test
        if m == "naive_week":
            sub = sub[sub["lead_days"] <= 6]
        sub = sub[sub[m].notna()]
        for grp in ["ALL", "zone"]:
            keys = ["lead_bucket"] if grp == "ALL" else ["zone", "lead_bucket"]
            for k, gs in sub.groupby(keys):
                zn = "ALL" if grp == "ALL" else k[0]
                hb = k if grp == "ALL" else k[1]
                err = (gs[m] - gs["target_price_sek_kwh"]).abs()
                ev_rows.append(("backtest", m, zn, hb, "mae_ore_kwh", float(err.mean() * 100), len(gs), now))
                ev_rows.append(("backtest", m, zn, hb, "hit_rate", float((err <= 0.02).mean()), len(gs), now))
    print(f"[fold {i}] test {test_start:%Y-%m-%d} -> {test_end:%Y-%m-%d} holdout={is_hold} n={len(test)}")

print("=== MAE (ore/kWh) by model x bucket, ALL zones, mean over folds ===")
tmp = pd.DataFrame(ev_rows, columns=["analysis","name","zone","hb","metric","value","n","run_at"])
print((tmp[(tmp.zone == "ALL") & (tmp.metric == "mae_ore_kwh")]
       .groupby(["name", "hb"])["value"].mean().round(2).unstack(fill_value="")))

## 5 · Prediction bands + horizon method
Fits on the pre-holdout window and derives p10/p50/p90 residual bands per
lead-bucket (these become the forecast uncertainty). Then decides the best way
to produce weekly/monthly forecasts — aggregating daily predictions vs. a direct
aggregate model — and records which wins.

In [ ]:
train_h = labeled[labeled["target_date"] < hold_start]
m_hold = fit(train_h, FEATS, "target_price_sek_kwh")
hold_eval = labeled[labeled["target_date"] >= hold_start].copy()
hold_eval["pred"] = m_hold.predict(hold_eval[FEATS])
hold_eval["resid"] = hold_eval["target_price_sek_kwh"] - hold_eval["pred"]
quant = {b: tuple(q) for b, q in
         hold_eval.groupby("lead_bucket")["resid"].quantile([0.1, 0.5, 0.9]).unstack().iterrows()}
assert set(quant) == {"1", "2-8", "9-62"}, f"FAIL: buckets {set(quant)}"
past = hold_eval[hold_eval["lead_days"] == 1]
mae = (past["resid"]).abs().mean()
hit = (past["resid"].abs() <= 0.02).mean()
print(f"[past check - out of sample] n={len(past)} MAE={mae*100:.2f} ore/kWh hit_within_2ore={hit:.1%}")
print("[bands]", {k: tuple(round(x, 4) for x in v) for k, v in quant.items()})


def agg_frame(df, hname):
    ps_col, pe_col = (("week_start", "week_end") if hname == "next-week"
                      else ("month_start", "month_end"))
    r = df[(df["target_date"] >= df[ps_col]) & (df["target_date"] <= df[pe_col])].copy()
    agg = r.groupby(["zone", "cutoff_date"]).agg(
        target_agg=("target_price_sek_kwh", "mean"),
        price_prevday=("price_prevday", "first"), price_mean_prev7=("price_mean_prev7", "first"),
        temp_prevday=("temp_prevday", "first"), temp_max_prevday=("temp_max_prevday", "first"),
        wind_prevday=("wind_prevday", "first"), precip_prevday=("precip_prevday", "first"),
        discharge_prevday=("discharge_prevday", "first"),
        gas_price_sek_kwh=("gas_price_sek_kwh", "first"), zone_idx=("zone_idx", "first"),
        _ps=(ps_col, "first"), _pe=(pe_col, "first")).reset_index()
    agg["dow_of_cutoff"] = agg["cutoff_date"].dt.weekday
    agg["month_of_period"] = pd.to_datetime(agg["_ps"]).dt.month
    return agg

best, quant_agg = {}, {}
for hname in ["next-week", "next-month"]:
    ps_col, pe_col = (("week_start", "week_end") if hname == "next-week"
                      else ("month_start", "month_end"))
    rows = labeled[(labeled["target_date"] >= labeled[ps_col])
                   & (labeled["target_date"] <= labeled[pe_col])
                   & (labeled[ps_col] >= hold_start) & (labeled[pe_col] <= mx)].copy()
    rows["pred"] = m_hold.predict(rows[FEATS])
    wtest = rows.groupby(["zone", "cutoff_date"]).agg(
        pred_agg=("pred", "mean"), actual_agg=("target_price_sek_kwh", "mean")).reset_index()
    mae_agg = (wtest["pred_agg"] - wtest["actual_agg"]).abs().mean() * 100

    agg = agg_frame(labeled, hname)
    agg = agg[agg["_pe"] <= mx]
    at, ae = agg[agg["_pe"] < hold_start], agg[agg["_ps"] >= hold_start]
    if len(at) > 100 and len(ae) > 0:
        md_ = fit(at, FEATS_AGG, "target_agg")
        ae = ae.copy()
        ae["p"] = md_.predict(ae[FEATS_AGG])
        mae_direct = (ae["p"] - ae["target_agg"]).abs().mean() * 100
    else:
        mae_direct = float("inf")
    winner = "agg_of_daily" if mae_agg <= mae_direct else f"direct_{hname.split('-')[1]}"
    best[hname] = winner
    resid = ((wtest["actual_agg"] - wtest["pred_agg"]) if winner == "agg_of_daily"
             else (ae["target_agg"] - ae["p"]))
    quant_agg[hname] = tuple(resid.quantile([0.1, 0.5, 0.9]))
    ev_rows.append(("backtest_agg", "agg_of_daily", "ALL", hname, "mae_ore_kwh",
                    float(mae_agg), len(wtest), now))
    ev_rows.append(("backtest_agg", winner, "ALL", hname, "mae_ore_kwh",
                    float(mae_direct), len(ae) if mae_direct != float("inf") else 0, now))
    print(f"[agg] {hname}: agg={mae_agg:.2f} ore direct={mae_direct:.2f} ore -> {winner}")

## 6 · Drivers (what moves the price)
Three diagnostics: deseasoned Pearson correlation of price vs. each driver,
permutation importance of every feature, and ablation (drop a feature group and
measure the MAE increase). Feeds `gold.price_model_evidence` so the site can show
*why* the forecast moves with weather / hydro / gas.

In [ ]:
gday = gas.copy()
gday["d"] = pd.to_datetime(gday["gd"])
gday = gday.rename(columns={"gp": "gas_asof"})[["d", "gas_asof"]]
dcorr = day.merge(gday, on="d", how="left")
dcorr["gm"] = dcorr["d"].dt.month
dcorr["p_a"] = dcorr["price_sek_kwh_avg"] - dcorr.groupby(["zone", "gm"])["price_sek_kwh_avg"].transform("mean")

for dname, col in [("temp_now", "temp_c_avg"), ("wind_now", "wind_speed_ms_avg"),
                   ("precip_now", "precipitation_mm_sum"), ("discharge_now", "discharge_m3s"),
                   ("gas_asof", "gas_asof"),
                   ("temp_prevday", "temp_prevday"), ("wind_prevday", "wind_prevday"),
                   ("discharge_prevday", "discharge_prevday")]:
    dcorr["_x"] = dcorr[col] - dcorr.groupby(["zone", "gm"])[col].transform("mean")
    for zn in ["ALL", "SE1", "SE2", "SE3", "SE4"]:
        sub = dcorr if zn == "ALL" else dcorr[dcorr["zone"] == zn]
        pair = sub[["p_a", "_x"]].dropna()
        if len(pair) >= 100:
            r = pair["p_a"].corr(pair["_x"])
            if pd.notna(r):
                ev_rows.append(("correlation", dname, zn, None, "pearson_r_deseasoned",
                                float(r), len(pair), now))
print("[corr] done")

model_full = fit(labeled, FEATS, "target_price_sek_kwh")
samp = hold_eval.sample(n=min(20000, len(hold_eval)), random_state=SEED)
pi = permutation_importance(model_full, samp[FEATS], samp["target_price_sek_kwh"],
                            n_repeats=3, random_state=SEED, scoring="neg_mean_absolute_error")
for fname, v in zip(FEATS, pi.importances_mean):
    ev_rows.append(("importance", fname, "ALL", None, "gbt_importance",
                    float(v * 100), len(samp), now))
print("[importance] done")

train_ab = labeled[labeled["target_date"] < hold_start]
test_ab = hold_eval
def hold_mae(cols):
    m = fit(train_ab, cols, "target_price_sek_kwh")
    return (test_ab["target_price_sek_kwh"] - m.predict(test_ab[cols])).abs().mean()

base_mae = hold_mae(FEATS)
ev_rows.append(("ablation", "all_features", "ALL", None, "mae_delta_ore", 0.0, len(test_ab), now))
GROUPS = {"weather": WEATHER_COLS, "hydro": ["discharge_prevday"], "gas": ["gas_price_sek_kwh"]}
for gname, cols in GROUPS.items():
    keep = [c for c in FEATS if c not in cols]
    d = (hold_mae(keep) - base_mae) * 100
    ev_rows.append(("ablation", gname, "ALL", None, "mae_delta_ore", float(d), len(test_ab), now))
    print(f"[ablation] drop {gname}: MAE +{d:.2f} ore/kWh")
print("Reading: positive mae_delta = that driver earns its keep.")

## 7 · Forecast + calendar summary
Applies the full model to the latest cutoff for each zone: 62 days ahead with
p10/p90 bands (`daily`), plus next-day / next-week / next-month summary rows
(`summary`) using the winning horizon method from cell 5.

In [ ]:
zcut = (sk.groupby("zone")["cutoff_date"].max()
        .reset_index().rename(columns={"cutoff_date": "zcut"}))
future = sk.merge(zcut, on="zone")
future = (future[(future["cutoff_date"] == future["zcut"])
                 & (future["target_price_sek_kwh"].isna())]
          .drop(columns=["zcut"]).copy())
future["pred"] = model_full.predict(future[FEATS])
q = future["lead_bucket"].map({b: quant[b] for b in quant})
future["q10"] = q.map(lambda t: t[0]); future["q50"] = q.map(lambda t: t[1]); future["q90"] = q.map(lambda t: t[2])
future["is_forecast"] = True
future["actual_sek_kwh"] = np.nan
future["spot_sek_kwh"] = future["pred"] + future["q50"]
future["spot_p10_sek_kwh"] = future["pred"] + future["q10"]
future["spot_p90_sek_kwh"] = future["pred"] + future["q90"]

past = hold_eval[hold_eval["lead_days"] == 1].copy()
qq = quant["1"]
past["is_forecast"] = False
past["actual_sek_kwh"] = past["target_price_sek_kwh"]
past["spot_sek_kwh"] = past["pred"] + qq[1]
past["spot_p10_sek_kwh"] = past["pred"] + qq[0]
past["spot_p90_sek_kwh"] = past["pred"] + qq[2]

daily = pd.concat([future, past])
daily["generated_at"] = dt.datetime.now().isoformat(sep=" ", timespec="seconds")

direct_models = {}
for hname in ["next-week", "next-month"]:
    if best[hname] != "agg_of_daily":
        agg = agg_frame(labeled, hname)
        direct_models[hname] = fit(agg[agg["_pe"] <= mx], FEATS_AGG, "target_agg")

summary = []
for zn in sorted(daily["zone"].unique()):
    zf = future[future["zone"] == zn]
    zcut_d = zf["cutoff_date"].max()
    next_day = zcut_d + pd.Timedelta(days=1)
    wk_start = zcut_d - pd.Timedelta(days=zcut_d.weekday()) + pd.Timedelta(days=7)
    wk_end = wk_start + pd.Timedelta(days=6)
    mo_start = zcut_d + pd.offsets.MonthBegin(1)
    mo_end = mo_start + pd.offsets.MonthEnd(1)
    nd = zf[zf["lead_days"] == 1].iloc[0]
    summary.append((zn, "next-day", next_day.date(), next_day.date(), "agg_of_daily",
                    nd["spot_sek_kwh"], nd["spot_p10_sek_kwh"], nd["spot_p90_sek_kwh"]))
    for hname, ps, pe in [("next-week", wk_start, wk_end), ("next-month", mo_start, mo_end)]:
        wq = quant_agg[hname]
        if best[hname] == "agg_of_daily":
            raw = zf[(zf["target_date"] >= ps) & (zf["target_date"] <= pe)]["pred"].mean()
            assert not np.isnan(raw), f"FAIL: empty window {hname} {zn}"
            pt = raw + wq[1]
            used = "agg_of_daily"
        else:
            one = sk[(sk["zone"] == zn) & (sk["cutoff_date"] == zcut_d)].head(1).copy()
            one["dow_of_cutoff"] = zcut_d.weekday()
            one["month_of_period"] = ps.month
            pt = direct_models[hname].predict(one[FEATS_AGG])[0] + wq[1]
            used = best[hname]
        summary.append((zn, hname, ps.date(), pe.date(), used,
                        pt, pt + (wq[0] - wq[1]), pt + (wq[2] - wq[1])))

summary = pd.DataFrame(summary, columns=["zone", "horizon", "period_start", "period_end",
                                         "model_used", "spot_sek_kwh", "spot_p10_sek_kwh", "spot_p90_sek_kwh"])
print(f"[forecast] daily={len(daily)} summary={len(summary)}")

## 8 · Bill + push + DQ checks
Converts spot prices to the consumer bill — `(spot + tax + fee) × 1.25`, with tax
and fee carried forward as-of the date. Pushes the three tables to
`gold.price_forecast`, `gold.price_forecast_summary`, `gold.price_model_evidence`
(delete + insert, so re-runs are idempotent), then asserts the DQ gates and
prints the end results in öre/kWh.

In [ ]:
def bill(df, date_col):
    tax = cp[(cp.table_name == "hushall") & (cp.customer_category == TAX_CATEGORY)][
        ["period_start", "elskatt_sek_kwh"]].rename(
        columns={"period_start": "valid_from", "elskatt_sek_kwh": "energy_tax_sek_kwh"})
    fee = cp[(cp.table_name == "elnat") & (cp.customer_category == FEE_CUSTOMER) & (cp.weighting == FEE_WEIGHTING)][
        ["period_start", "elnatspris_sek_kwh"]].rename(
        columns={"period_start": "valid_from", "elnatspris_sek_kwh": "grid_fee_sek_kwh"})
    out = df.copy()
    for c in ["energy_tax_sek_kwh", "grid_fee_sek_kwh",
              "total_bill_sek_kwh", "total_bill_p10_sek_kwh", "total_bill_p90_sek_kwh", "_d"]:
        if c in out.columns:
            out = out.drop(columns=[c])
    out["_d"] = pd.to_datetime(out[date_col])
    out = out.sort_values("_d").reset_index(drop=True)
    for t in (tax, fee):
        t = t.copy()
        t["valid_from"] = pd.to_datetime(t["valid_from"])
        t = t.sort_values("valid_from").drop_duplicates("valid_from", keep="last")
        out = pd.merge_asof(out, t, left_on="_d", right_on="valid_from",
                            allow_exact_matches=True).drop(columns=["valid_from"])
    out["total_bill_sek_kwh"] = (out["spot_sek_kwh"] + out["energy_tax_sek_kwh"] + out["grid_fee_sek_kwh"]) * (1 + VAT_RATE)
    out["total_bill_p10_sek_kwh"] = (out["spot_p10_sek_kwh"] + out["energy_tax_sek_kwh"] + out["grid_fee_sek_kwh"]) * (1 + VAT_RATE)
    out["total_bill_p90_sek_kwh"] = (out["spot_p90_sek_kwh"] + out["energy_tax_sek_kwh"] + out["grid_fee_sek_kwh"]) * (1 + VAT_RATE)
    return out.drop(columns=["_d"])

daily = bill(daily, "target_date")
summary = bill(summary, "period_start")
summary["generated_at"] = daily["generated_at"].iloc[0]
daily["obs_year"] = pd.to_datetime(daily["target_date"]).dt.year
evid = pd.DataFrame(ev_rows, columns=["analysis","name","zone","horizon_bucket",
                                      "metric_name","metric_value","n","run_at"])


def push(df, table, cols):
    sub = df[cols].where(pd.notna(df[cols]), None)
    with ENGINE.begin() as conn:
        conn.execute(text(f"DELETE FROM {table}"))
        for i in range(0, len(sub), 40):
            values = ",\n".join("(" + ",".join(lit(v) for v in row) + ")"
                                for row in sub.iloc[i:i+40].itertuples(index=False, name=None))
            conn.execute(text(f"INSERT INTO {table} ({','.join(cols)}) VALUES {values}"))
    print(f"[push] {table}: {len(df)} rows")


with ENGINE.begin() as conn:
    conn.execute(text("""CREATE TABLE IF NOT EXISTS gold.price_forecast (
      zone text, target_date date, lead_days int, lead_bucket text, generated_at timestamp,
      is_forecast boolean, spot_sek_kwh double precision, spot_p10_sek_kwh double precision,
      spot_p90_sek_kwh double precision, actual_sek_kwh double precision,
      energy_tax_sek_kwh double precision, grid_fee_sek_kwh double precision,
      total_bill_sek_kwh double precision, total_bill_p10_sek_kwh double precision,
      total_bill_p90_sek_kwh double precision, obs_year int)"""))
    conn.execute(text("""CREATE TABLE IF NOT EXISTS gold.price_forecast_summary (
      zone text, horizon text, period_start date, period_end date, generated_at timestamp,
      model_used text, spot_sek_kwh double precision, spot_p10_sek_kwh double precision,
      spot_p90_sek_kwh double precision, energy_tax_sek_kwh double precision,
      grid_fee_sek_kwh double precision, total_bill_sek_kwh double precision,
      total_bill_p10_sek_kwh double precision, total_bill_p90_sek_kwh double precision)"""))
    conn.execute(text("""CREATE TABLE IF NOT EXISTS gold.price_model_evidence (
      analysis text, name text, zone text, horizon_bucket text, metric_name text,
      metric_value double precision, n bigint, run_at timestamp)"""))

dcols = ["zone", "target_date", "lead_days", "lead_bucket", "generated_at", "is_forecast",
         "spot_sek_kwh", "spot_p10_sek_kwh", "spot_p90_sek_kwh", "actual_sek_kwh",
         "energy_tax_sek_kwh", "grid_fee_sek_kwh",
         "total_bill_sek_kwh", "total_bill_p10_sek_kwh", "total_bill_p90_sek_kwh", "obs_year"]
scols = ["zone", "horizon", "period_start", "period_end", "generated_at", "model_used",
         "spot_sek_kwh", "spot_p10_sek_kwh", "spot_p90_sek_kwh",
         "energy_tax_sek_kwh", "grid_fee_sek_kwh",
         "total_bill_sek_kwh", "total_bill_p10_sek_kwh", "total_bill_p90_sek_kwh"]
ecols = ["analysis", "name", "zone", "horizon_bucket", "metric_name", "metric_value", "n", "run_at"]

push(daily, "gold.price_forecast", dcols)
push(summary, "gold.price_forecast_summary", scols)
push(evid, "gold.price_model_evidence", ecols)

fut = daily[daily["is_forecast"]]
assert len(fut) == MAX_LEAD * 4, f"FAIL: future rows {len(fut)}"
assert len(summary) == 12, f"FAIL: summary rows {len(summary)}"
assert fut["spot_sek_kwh"].notna().all(), "FAIL: null predictions"
print("[ok] DQ gates passed")

print("=== THE END RESULTS (ore/kWh) ===")
show = summary.copy()
for c in ["spot_sek_kwh", "spot_p10_sek_kwh", "spot_p90_sek_kwh", "total_bill_sek_kwh"]:
    show[c] = (show[c] * 100).round(1)
print(show[["zone", "horizon", "period_start", "period_end", "model_used",
             "spot_sek_kwh", "spot_p10_sek_kwh", "spot_p90_sek_kwh", "total_bill_sek_kwh"]].to_string(index=False))